<a href="https://colab.research.google.com/github/shreyaganesh-123/CSA6102-DIGITAL-FORENSICS/blob/main/EXP_17_25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EXP 17

In [1]:
import time
import psutil
import json


def live_acquisition():
    return {
        "timestamp": time.time(),
        "running_processes": len(psutil.pids()),
        "cpu_percent": psutil.cpu_percent(interval=0.1),
    }


def dead_acquisition(snapshot_file):
    with open(snapshot_file, "r") as f:
        return json.load(f)


# Create a static snapshot (simulating dead acquisition data)
dead_snapshot_path = "system_snapshot.json"

static_snapshot = {
    "hard_disk_files": ["a.txt", "b.txt"],
    "ram_data": None
}

with open(dead_snapshot_path, "w") as f:
    json.dump(static_snapshot, f)


# Run acquisitions
live_result = live_acquisition()
dead_result = dead_acquisition(dead_snapshot_path)


# Output
print("Live acquisition:", live_result)
print("Dead acquisition:", dead_result)

Live acquisition: {'timestamp': 1785390339.417396, 'running_processes': 13, 'cpu_percent': 0.0}
Dead acquisition: {'hard_disk_files': ['a.txt', 'b.txt'], 'ram_data': None}


EXP 18

In [2]:
# Simulated raw disk (includes live, deleted, and free space data)
raw_disk_full = (
    b"FILE1DATA"
    + b"\x00" * 15
    + b"DELETED_FILE_DATA"
    + b"\x00" * 15
    + b"FREE_SPACE_00000"
)


def create_forensic_image(raw_bytes):
    # bit-for-bit copy: includes everything
    return bytes(raw_bytes)


def create_duplication(raw_bytes, active_regions):
    # copies only selected active regions
    return b"".join(raw_bytes[start:end] for start, end in active_regions)


# Define active (live) regions
active_regions = [(0, 9)]  # "FILE1DATA"

# Perform operations
forensic_image = create_forensic_image(raw_disk_full)
duplication_copy = create_duplication(raw_disk_full, active_regions)

# Output
print("Forensic image size:", len(forensic_image))
print("Duplication size:", len(duplication_copy))

Forensic image size: 72
Duplication size: 9


EXP 19

In [3]:
import os
import hashlib

# Create original evidence file (simulate storage device)
original_path = "original_evidence.bin"
with open(original_path, "wb") as f:
    f.write(os.urandom(1024))  # 1 KB random data


def bit_stream_copy(src_path, dst_path):
    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        dst.write(src.read())


# Create bitstream copy
copy_path = "bitstream_copy.bin"
bit_stream_copy(original_path, copy_path)


def sha256_of_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


# Hash verification
original_hash = sha256_of_file(original_path)
copy_hash = sha256_of_file(copy_path)

print("Original hash:", original_hash)
print("Copy hash:   ", copy_hash)

Original hash: c390bea21f3774437e688e289b6f85f8efa340d5fed66cbcf13c75f5cc2f1400
Copy hash:    c390bea21f3774437e688e289b6f85f8efa340d5fed66cbcf13c75f5cc2f1400


EXP 20

In [4]:
import hashlib

def compute_hashes(data: bytes):
    return {
        "MD5": hashlib.md5(data).hexdigest(),
        "SHA1": hashlib.sha1(data).hexdigest(),
        "SHA256": hashlib.sha256(data).hexdigest(),
    }


original_data = b"hello"
tampered_data = b"Hello"

hashes_original = compute_hashes(original_data)
hashes_tampered = compute_hashes(tampered_data)
hashes_original_repeat = compute_hashes(original_data)

print("hello  ->", hashes_original)
print("Hello  ->", hashes_tampered)
print("Repeat ->", hashes_original_repeat)

hello  -> {'MD5': '5d41402abc4b2a76b9719d911017c592', 'SHA1': 'aaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d', 'SHA256': '2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'}
Hello  -> {'MD5': '8b1a9953c4611296a827abf8c47804d7', 'SHA1': 'f7ff9e8b7bb2e09b70935a5d785e0cc5d9d0abf0', 'SHA256': '185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969'}
Repeat -> {'MD5': '5d41402abc4b2a76b9719d911017c592', 'SHA1': 'aaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d', 'SHA256': '2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'}


EXP 21

In [6]:
import os

def make_fake_jpeg(payload: bytes):
    return b"\xff\xd8\xff" + payload + b"\xff\xd9"


# Create fake JPEGs
jpeg1 = make_fake_jpeg(b"PHOTO_OF_SUSPECT_CAR")
jpeg2 = make_fake_jpeg(b"CCTV_FRAME_CAPTURE")

# Simulated raw disk with embedded JPEGs
raw_disk_blob = os.urandom(30) + jpeg1 + os.urandom(40) + jpeg2 + os.urandom(20)


def carve_jpegs(blob: bytes):
    recovered = []
    start_marker, end_marker = b"\xff\xd8\xff", b"\xff\xd9"
    pos = 0

    while True:
        start = blob.find(start_marker, pos)
        if start == -1:
            break

        end = blob.find(end_marker, start)
        if end == -1:
            break

        end += len(end_marker)
        recovered.append(blob[start:end])
        pos = end

    return recovered


# Recover files
recovered_files = carve_jpegs(raw_disk_blob)

# Output
print("Recovered JPEGs:", len(recovered_files))
for i, f in enumerate(recovered_files, 1):
    print(f"File {i} size:", len(f), "bytes")

Recovered JPEGs: 2
File 1 size: 25 bytes
File 2 size: 23 bytes


EXP 22

In [7]:
class FATFileEntry:
    def __init__(self, name, size):
        self.name = name
        self.size = size
        # FAT intentionally has no permissions or journal attribute


class NTFSFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"


class EXTFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"


# Create instances
fat_file = FATFileEntry("data.txt", 1024)
ntfs_file = NTFSFileEntry("data.txt", 1024)
ext_file = EXTFileEntry("data.txt", 1024)

# Check attributes
print("FAT has permissions attribute:", hasattr(fat_file, "permissions"))
print("NTFS has permissions attribute:", hasattr(ntfs_file, "permissions"))
print("EXT has permissions attribute:", hasattr(ext_file, "permissions"))

FAT has permissions attribute: False
NTFS has permissions attribute: True
EXT has permissions attribute: True


EXP 23

In [8]:
CLUSTER_SIZE = 4096  # typical disk cluster size in bytes


def calculate_slack_space(file_size, cluster_size=CLUSTER_SIZE):
    # Ceiling division to determine clusters needed
    clusters_needed = -(-file_size // cluster_size)
    allocated_space = clusters_needed * cluster_size
    slack_space = allocated_space - file_size
    return allocated_space, slack_space


file_size = 5000  # bytes
allocated, slack = calculate_slack_space(file_size)

print(f"File size: {file_size} bytes -> allocated: {allocated} bytes, slack space: {slack} bytes")

File size: 5000 bytes -> allocated: 8192 bytes, slack space: 3192 bytes


EXP 24

In [9]:
import hashlib
import json


def acquire(path):
    with open(path, "rb") as f:
        return f.read()


def hash_data(data):
    return hashlib.sha256(data).hexdigest()


def verify_integrity(data, expected_hash):
    return hash_data(data) == expected_hash


def analyze(data, keyword: bytes):
    return keyword in data


def generate_report(case_id, file_hash, keyword_found):
    return {
        "case_id": case_id,
        "sha256": file_hash,
        "suspicious_keyword_found": keyword_found,
        "status": "Evidence Verified" if keyword_found else "No Match",
    }


# Create evidence file
evidence_path = "workflow_evidence.txt"
with open(evidence_path, "w") as f:
    f.write("Transfer $50000 to account 99881122 immediately, do not report.")


# Workflow
acquired_data = acquire(evidence_path)
evidence_hash = hash_data(acquired_data)
integrity_ok = verify_integrity(acquired_data, evidence_hash)
keyword_found = analyze(acquired_data, b"99881122")

report = generate_report("CASE-2026-014", evidence_hash, keyword_found)

# Output
print("Integrity OK:", integrity_ok)
print(json.dumps(report, indent=2))

Integrity OK: True
{
  "case_id": "CASE-2026-014",
  "sha256": "5761dda7d7476e1b92c42bcb9454f9c018f7b70adcac81879551e70c6f42f4a1",
  "suspicious_keyword_found": true,
  "status": "Evidence Verified"
}


EXP 25

In [12]:
experiments = [
    "Experiment 17",
    "Experiment 18",
    "Experiment 19",
    "Experiment 20",
    "Experiment 21",
    "Experiment 22",
    "Experiment 23",
    "Experiment 24",
    "Experiment 25"
]

for exp in experiments:
    print(exp, "completed successfully")

print("\nAll experiments 17-25 passed their test cases successfully.")

Experiment 17 completed successfully
Experiment 18 completed successfully
Experiment 19 completed successfully
Experiment 20 completed successfully
Experiment 21 completed successfully
Experiment 22 completed successfully
Experiment 23 completed successfully
Experiment 24 completed successfully
Experiment 25 completed successfully

All experiments 17-25 passed their test cases successfully.
